In [1]:
import os
import pandas as pd
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home'

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("test").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 21:44:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
#required files
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 21:44:45--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2684:200:b:20a5:b140:21, 2600:9000:2684:2000:b:20a5:b140:21, 2600:9000:2684:4200:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2684:200:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  38.2MB/s    in 1.8s    

2026-03-09 21:44:47 (38.2 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]

--2026-03-09 21:44:48--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2684:200:b:20a5:b140:21, 2600:9000:2684:2000:b:20a5:b140:21, 2600:9000:2684:4200:b:20a5:b140

In [3]:
#read the parquet file
yellow_df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

#partition into 4 parts and write to parquet
yellow_df.repartition(4).write.parquet('yellow_tripdata/test')

#folder with parquet files
folder_path = 'yellow_tripdata/test'
files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]

sizes = [os.path.getsize(os.path.join(folder_path, f)) for f in files]
avg_size_mb = sum(sizes) / len(sizes) / (1024 * 1024)
print(f'Average file size: {avg_size_mb:.2f} MB')

Average file size: 24.42 MB


In [4]:
from pyspark.sql import functions as F
#how many trips on the 15th of November?
yellow_df.filter(F.dayofmonth('tpep_pickup_datetime') == 15).count()

162604

In [5]:
#length of longest trip in hours
yellow_df.select(F.max((F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600)).show()

+--------------------------------------------------------------------------------------------------------------------------------------+
|max(((unix_timestamp(tpep_dropoff_datetime, yyyy-MM-dd HH:mm:ss) - unix_timestamp(tpep_pickup_datetime, yyyy-MM-dd HH:mm:ss)) / 3600))|
+--------------------------------------------------------------------------------------------------------------------------------------+
|                                                                                                                     90.64666666666666|
+--------------------------------------------------------------------------------------------------------------------------------------+



In [7]:
zone_lookup_df = spark.read.csv('taxi_zone_lookup.csv', header=True)
zone_lookup_df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [9]:
#find the name of the least frequent pickup location by joining with the zone lookup table
least_freq_location_id = yellow_df.groupBy('PULocationID').count().orderBy('count').first()['PULocationID']
zone_lookup_df.where(zone_lookup_df.LocationID == least_freq_location_id).show()

+----------+---------+--------------------+------------+
|LocationID|  Borough|                Zone|service_zone|
+----------+---------+--------------------+------------+
|       105|Manhattan|Governor's Island...| Yellow Zone|
+----------+---------+--------------------+------------+

